# 02_show_footprints_fast

Tutorial for large-scale (1–5M polygon) footprint/building visualization using:
- **Track 1**: Interactive, GPU-accelerated maps via Lonboard/deck.gl.
- **Track 2**: CPU-only raster fallbacks via Datashader (for headless environments).

### Requirements
Requires the `viz-fast` packages:
```bash
pip install -e ".[viz-fast]"
# or
conda env update -f environment.yml
```

Since Jupyter kernels cannot automatically detect browser WebGL2 support, choose the track that fits your environment.

In [ ]:
from openplaces.viz import show_entities_interactive, show_entities_raster

# The recipe to visualize (curated footprints). If you need to fallback
# to the evidence-only spine, use 'US_footprint-spine-2026'.
RECIPE = 'US_footprint-cheer-2026'
ONE_COUNTY = 'US-NC-CE'  # Carteret County, NC
STATE = 'US-NC'  # North Carolina (~44 of 100 counties processed)

# Track 1 — Interactive GPU rendering (Lonboard/deck.gl)

Uses `SolidPolygonLayer` (avoiding the slower composite `PolygonLayer`).
The map renders below and supports pan, zoom, tilt, and rotation.

If `color_by` matches a palette, a separate legend widget displays
above the map (deck.gl's WebGL canvas does not support direct DOM overlays).

In [ ]:
# One county colored by occupancy_type (canonical classification).
show_entities_interactive(RECIPE, admin_id=ONE_COUNTY, color_by='occupancy_type')

## Viewport-bounded loading (`bbox`)

`get_entities` accepts a `bbox` in EPSG:4326, using Parquet predicate
pushdown on the geometry bounding boxes. Non-overlapping files contribute
no rows, making multi-file spatial queries highly efficient.

Running on the full state with `missing='ignore'` displays only the
currently processed counties, showing the 1–5M polygon scale.

In [ ]:
# Load a subset of a single county's file via spatial pushdown.
coastal_bbox = (-76.75, 34.65, -76.55, 34.85)
show_entities_interactive(RECIPE, admin_id=ONE_COUNTY, bbox=coastal_bbox)

In [ ]:
# Full state: ~2.7M polygons. Skips unprocessed counties via
# missing='ignore'.
show_entities_interactive(
    RECIPE, admin_id=STATE, color_by='occupancy_type', missing='ignore'
)

# Track 2 — CPU raster fallback (Datashader)

A fallback for headless/CPU-only environments without WebGL2 support. It
rasterizes server-side to a static image (re-run cell to pan/zoom). Data
is reprojected to EPSG:3857 before rendering to avoid aspect ratio
distortion.

Legends are drawn directly onto the image. Setting `show_boundaries=True`
overlays aligned county boundaries using an additional line-drawing pass.

In [ ]:
# Plain density heatmap (no color_by) with county boundaries.
show_entities_raster(RECIPE, admin_id=ONE_COUNTY, show_boundaries=True)

In [ ]:
# Full state colored by occupancy_type, including county outlines.
show_entities_raster(
    RECIPE,
    admin_id=STATE,
    color_by='occupancy_type',
    missing='ignore',
    plot_width=1200,
    plot_height=800,
    show_boundaries=True,
)